# IoT sensor data extraction

This notebook aims to simulate a datasource consisting of an IoT device that captures and sends real-time data.

In particular, this imaginary device represents a small sensor placed in a parking slot that sends an event when a vehicle arrives or leaves so that people will be able to view free parking slots in real time.

## Devices list

The following list is an example containing ids of sensors. This can be crossed with data about parking lots in Barcelona in order to obtain realistic locations:
[https://opendata-ajuntament.barcelona.cat/data/ca/dataset/aparcaments-bcn](https://opendata-ajuntament.barcelona.cat/data/ca/dataset/aparcaments-bcn)

In [7]:
# 2. Simulación de la "Semilla" Estructurada (CSV en memoria)
# En tu proyecto, aquí usarías la librería 'csv' o 'pandas' para cargar miles de plazas reales.
#devices_bcn = ["PZ-001", "PZ-002", "PZ-003", "PZ-004", "PZ-005"]

devices_bcn = [f"P_{i}" for i in range(10000)]
print(devices_bcn)


['P_0', 'P_1', 'P_2', 'P_3', 'P_4', 'P_5', 'P_6', 'P_7', 'P_8', 'P_9', 'P_10', 'P_11', 'P_12', 'P_13', 'P_14', 'P_15', 'P_16', 'P_17', 'P_18', 'P_19', 'P_20', 'P_21', 'P_22', 'P_23', 'P_24', 'P_25', 'P_26', 'P_27', 'P_28', 'P_29', 'P_30', 'P_31', 'P_32', 'P_33', 'P_34', 'P_35', 'P_36', 'P_37', 'P_38', 'P_39', 'P_40', 'P_41', 'P_42', 'P_43', 'P_44', 'P_45', 'P_46', 'P_47', 'P_48', 'P_49', 'P_50', 'P_51', 'P_52', 'P_53', 'P_54', 'P_55', 'P_56', 'P_57', 'P_58', 'P_59', 'P_60', 'P_61', 'P_62', 'P_63', 'P_64', 'P_65', 'P_66', 'P_67', 'P_68', 'P_69', 'P_70', 'P_71', 'P_72', 'P_73', 'P_74', 'P_75', 'P_76', 'P_77', 'P_78', 'P_79', 'P_80', 'P_81', 'P_82', 'P_83', 'P_84', 'P_85', 'P_86', 'P_87', 'P_88', 'P_89', 'P_90', 'P_91', 'P_92', 'P_93', 'P_94', 'P_95', 'P_96', 'P_97', 'P_98', 'P_99', 'P_100', 'P_101', 'P_102', 'P_103', 'P_104', 'P_105', 'P_106', 'P_107', 'P_108', 'P_109', 'P_110', 'P_111', 'P_112', 'P_113', 'P_114', 'P_115', 'P_116', 'P_117', 'P_118', 'P_119', 'P_120', 'P_121', 'P_122', 'P

## Data generation

The code leverages asyncio library to simulate the execution of all the above devices independently. Each device awaits a randomly selected amount of time before changing its state (which can be "empty" or "full"), generating a new event and sending it to Kafka.

***TODO: Integrate kafka with the code***

In [2]:
KAFKA_BROKER = [
    'localhost:9092',
]

TOPIC_NAME = 'parking-lot-events'


In [ ]:
import json
import time
import random
import asyncio
from datetime import datetime
from kafka import KafkaProducer


def init_producer(bootstrap_servers: list[str]):
    return KafkaProducer(
        bootstrap_servers=bootstrap_servers,
        value_serializer=lambda v: json.dumps(v).encode('utf-8')
    )


async def simulate(device_id: str, producer: KafkaProducer, topic_name: str):
    is_empty = random.choice([True,False])
    try:
        while True:
            sleep = random.uniform(1.0, 10.0)
            await asyncio.sleep(sleep)

            is_empty = not is_empty

            event = {
                "device_id": device_id,
                "timestamp": datetime.utcnow().isoformat(),
                "state": "empty" if is_empty else "full",
            }
            
            producer.send(topic_name, value=event)
            producer.flush()
            print(f"[{event['timestamp']}] Event generated: Lot {event['device_id']} now is {event['state']}.")
            
    except KeyboardInterrupt:
        print("\nSimulation stopped by user.")
    finally:
        producer.flush()
        producer.close()


async def main():
    current_tasks = []
        
    kafka_producer = init_producer(KAFKA_BROKER)

    for device in devices_bcn:
        task = asyncio.create_task(simulate(device, kafka_producer, TOPIC_NAME))
        current_tasks.append(task)

    await asyncio.gather(*current_tasks)


In [9]:
# asyncio.run(main())

await main()

/tmp/ipykernel_4808/2327283690.py:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


[2026-03-16T17:28:20.384859] Event generated: Lot P_1389 now is empty.
[2026-03-16T17:28:20.413502] Event generated: Lot P_337 now is full.
[2026-03-16T17:28:20.417042] Event generated: Lot P_3118 now is full.
[2026-03-16T17:28:20.419677] Event generated: Lot P_3600 now is full.
[2026-03-16T17:28:20.423126] Event generated: Lot P_1887 now is empty.
[2026-03-16T17:28:20.428709] Event generated: Lot P_4356 now is full.
[2026-03-16T17:28:20.433425] Event generated: Lot P_1930 now is empty.
[2026-03-16T17:28:20.436882] Event generated: Lot P_1901 now is full.
[2026-03-16T17:28:20.440785] Event generated: Lot P_2296 now is empty.
[2026-03-16T17:28:20.444655] Event generated: Lot P_6140 now is full.
[2026-03-16T17:28:20.447319] Event generated: Lot P_4361 now is empty.
[2026-03-16T17:28:20.449682] Event generated: Lot P_4941 now is empty.
[2026-03-16T17:28:20.451987] Event generated: Lot P_4095 now is empty.
[2026-03-16T17:28:20.454153] Event generated: Lot P_5782 now is full.
[2026-03-16T17

CancelledError: 